In [49]:
import requests

url = "http://localhost:8080/otp/gtfs/v1"

query = """
query
($origin: PlanLabeledLocationInput!,
$destination: PlanLabeledLocationInput!,
$modes: PlanModesInput!,
$preferences: PlanPreferencesInput,
$dateTime: PlanDateTimeInput)
{
  planConnection(
    origin: $origin
    destination: $destination
    dateTime: $dateTime
    modes: $modes
    preferences: $preferences

  ) {
    edges {
      node {
        start
        end
        legs {
          mode
          start
          end
          from {
            name
            lat
            lon

          }
          to {
            name
          }
          route {
            shortName
          }
        }
      }
    }
  }
}
"""



variables = {
    "origin": {
      "location": {
        "coordinate": {
          "latitude": 57.086712,
          "longitude": 9.955823
        }
      }
    },
    "destination": {
      "location": {
        "coordinate": {
          "latitude": 57.315421,
          "longitude": 10.523608
        }
      }
    },
    "dateTime": {
      "earliestDeparture": '2024-01-25T15:00:00+0100'
    },
    "modes": {
      "directOnly": True,
      "transitOnly": False,
      "direct": ["CAR"],
      "transit": {
        "transit": [
            { "mode": "RAIL"}, #, cost: { reluctance: 1 }
            { "mode": "BUS" },
            { "mode": "SUBWAY" }
        ]
      }
    },
    "preferences": {
      "transit": {
        "filters": [
          {
            "include": {
              "routeShortNames": ["M2"]
            }
          }
        ]
      }
    }
}


In [50]:
response = requests.post(
    url,
    json={
        "query": query,
        "variables": variables
    }
)
print(response.json())

{'errors': [{'message': "Validation error (SubselectionRequired@[planConnection/edges/node/legs/start]) : Subselection required for type 'LegTime!' of field 'start'", 'locations': [{'line': 23, 'column': 11}], 'extensions': {'classification': 'ValidationError'}}, {'message': "Validation error (SubselectionRequired@[planConnection/edges/node/legs/end]) : Subselection required for type 'LegTime!' of field 'end'", 'locations': [{'line': 24, 'column': 11}], 'extensions': {'classification': 'ValidationError'}}, {'message': "Validation error (SubselectionRequired@[planConnection/edges/node/legs/from/stop]) : Subselection required for type 'Stop' of field 'stop'", 'locations': [{'line': 29, 'column': 13}], 'extensions': {'classification': 'ValidationError'}}]}


In [51]:
import json

print(json.dumps(response.json(), indent=2))

{
  "errors": [
    {
      "message": "Validation error (SubselectionRequired@[planConnection/edges/node/legs/start]) : Subselection required for type 'LegTime!' of field 'start'",
      "locations": [
        {
          "line": 23,
          "column": 11
        }
      ],
      "extensions": {
        "classification": "ValidationError"
      }
    },
    {
      "message": "Validation error (SubselectionRequired@[planConnection/edges/node/legs/end]) : Subselection required for type 'LegTime!' of field 'end'",
      "locations": [
        {
          "line": 24,
          "column": 11
        }
      ],
      "extensions": {
        "classification": "ValidationError"
      }
    },
    {
      "message": "Validation error (SubselectionRequired@[planConnection/edges/node/legs/from/stop]) : Subselection required for type 'Stop' of field 'stop'",
      "locations": [
        {
          "line": 29,
          "column": 13
        }
      ],
      "extensions": {
        "classificatio

In [112]:
import pandas as pd

data = response.json()["data"]["planConnection"]["edges"]

rows = []
for i, edge in enumerate(data):
    node = edge["node"]
    for j, leg in enumerate(node["legs"]):
        route = leg.get("route") or {} #for WALK "shortName" is null

        rows.append({
            "start": node["start"],
            "end": node["end"],
            "iteration_id": i,
            "leg_id": j,
            "mode": leg["mode"],
            "start_time": leg["startTime"],
            "end_time": leg["endTime"],
            "from": leg["from"]["name"],
            "to": leg["to"]["name"],
            "route": route.get("shortName")
        })
df = pd.DataFrame(rows)

In [113]:
df

,start,end,itegration_id,leg_id,mode,start_time,end_time,from,to,route
0,2024-04-20T12:40:25+02:00,2024-04-20T13:43:06+02:00,0,0,WALK,1713609625000,1713610980000,Origin,Femøren St. (Metro),None
1,2024-04-20T12:40:25+02:00,2024-04-20T13:43:06+02:00,0,1,SUBWAY,1713610980000,1713611700000,Femøren St. (Metro),Nørreport St. (Metro),M2
2,2024-04-20T12:40:25+02:00,2024-04-20T13:43:06+02:00,0,2,WALK,1713611700000,1713613386000,Nørreport St. (Metro),Destination,None
3,2024-04-20T12:45:25+02:00,2024-04-20T13:48:06+02:00,1,0,WALK,1713609925000,1713611280000,Origin,Femøren St. (Metro),None
4,2024-04-20T12:45:25+02:00,2024-04-20T13:48:06+02:00,1,1,SUBWAY,1713611280000,1713612000000,Femøren St. (Metro),Nørreport St. (Metro),M2
5,2024-04-20T12:45:25+02:00,2024-04-20T13:48:06+02:00,1,2,WALK,1713612000000,1713613686000,Nørreport St. (Metro),Destination,None
6,2024-04-20T12:50:25+02:00,2024-04-20T13:53:06+02:00,2,0,WALK,1713610225000,1713611580000,Origin,Femøren St. (Metro),None
7,2024-04-20T12:50:25+02:00,2024-04-20T13:53:06+02:00,2,1,SUBWAY,1713611580000,1713612300000,Femøren St. (Metro),Nørreport St. (Metro),M2
8,2024-04-20T12:50:25+02:00,2024-04-20T13:53:06+02:00,2,2,WALK,1713612300000,1713613986000,Nørreport St. (Metro),Destination,None
9,2024-04-20T12:55:25+02:00,2024-04-20T13:58:06+02:00,3,0,WALK,1713610525000,1713611880000,Origin,Femøren St. (Metro),None
